# KCORC Summer School - TESPy Workshop

## Case Study: 5.5 MWe Double-stage ORC Kirchstockach

The Kirchstockach geothermal power plant is a two-stage Organic Rankine Cycle
(ORC) system consisting of a High-Temperature (HT) ORC and a Low-Temperature
(LT) ORC operating in series. The geothermal brine first transfers heat to the
HT cycle and subsequently to the LT cycle before reinjection.

![ORC flowsheet](../orc.svg)

Figure 1: Scheme of the double-stage ORC power plant in Kirchstockach, Germany
(Florian Heberle, Thomas Jahrfeld and Dieter Brüggemann, 2015)
https://worldgeothermal.org/pdf/IGAstandard/WGC/2015/26002.pdf


### Objective

Your task is to investigate the behavior of the geothermal ORC under off-design
operating conditions. Using the design-point model, evaluate the impact of
variations in boundary conditions on the plant's performance.

### Tasks

1. Review the design point model in the code below:
   
   - Run the simulation model and check the results.
   - Try to exchange specifications one by one and rerun it.
   - Make a list of all specifications, that cannot be controlled in part-load
     operation.

2. Convert the design-point model into an off-design model. First, prepare and
   test these correlations on single components. What are the assumptions are
   taken in the modeling of these correlations and what simplifications are
   done?

   - Integrate the data provided in the axial turbine off-design map with the
     help of a `UserDefinedEquation`. Validate your implementation against the
     data in a single component model.
   - Calculate the design point area of the heat exchangers based on the
     provided heat transfer coefficients per phase/section. Copy the area
     equation of `SectionedHeatExchanger` instances and implement a scaling
     of the heat transfer coefficients with hot side and cold side mass flow
     according to $\alpha_j = \alpha_{j,\text{design}} \cdot
\left(\frac{\dot m}{\dot m_\text{design}}\right)^{Re_{\text{exp},j}}$. Use a
     `UserDefinedEquation` to embed the equation in a single component model
     and validate the results.

3. Transfer the implemented methods into the LT-ORC model for each turbine and
   heat exchanger and test the execution of the off-design model at design
   point conditions. What outcome do you expect in design conditions?
4. Simulate variations in the split mass flow rate (LT mass flow rate from 30
   to 70 kg/s, assuming a constant total geothermal mass flow rate) and
   variations in the ambient temperature (from 0 to 30 °C). Evaluate the
   influence of these operating conditions on turbine power, net electrical
   power, thermal efficiency, and reinjection temperature.
5. What degrees of freedom exist in the plant off-design operation? Run 
   sensitivity analyses on one setting of geothermal conditions to identify, 
   how these affect to system performance. 
6. Repeat the analysis under varying ambient and geothermal conditions.
7. Implement the same changes for the full LT and HT system.

### Notes

#### Turbine map

The turbine map data in the json correlation provides data for one correlation
directly, i.e. the isentropic efficiency, which is a function of pressure ratio
(at optimal rotational speed).


A second correlation is defined by the geometry of the turbine, i.e. the mass
flow, which is a function of the inlet state and the throat area $A$.

$$
\dot m = C_{d} \cdot A_{th} \cdot \rho^* \cdot a^*
$$

The density $\rho^*$ is the density at the pressure $p^*$ where the isentropic
fluid velocity is equal to the speed of sound. To find that pressure, you need
to solve the following expression for p:

$$
\sqrt{2 \cdot \left(h_\text{in} - h\left(p^*,s_\text{in}\right)\right)} - a\left(p^*,s_\text{in}\right)
$$

The turbine data provided in the .json file is the simplified form of the map 
published by Jan Spale et al. (2026): "Design methodology and experimental
assessment of a small-scale supersonic axial impulse ORC turbine",
https://doi.org/10.1016/j.energy.2026.141901.

#### SectionedHeatExchanger Model

In the `SectionedHeatExchanger` class, the heat exchange is in counter flow
configuration and discretized in $N$ sections and $N+1$ section boundaries 
(or steps).

The heat exchanger's heat transfer $\dot Q$ and the hot and cold sides'
respective $\Delta h$ and $\Delta p$ distribute linearly over the sections,
therefore each section $j$ has same $\Delta h_{j}$ etc. Additional sections are
inserted at phase change locations according to the methodology defined in
Bell et al. (2015): "A generalized moving-boundary algorithm to predict the 
heat transfer rate of counterflow heat exchangers for any phase configuration", 
https://doi.org/10.1016/j.applthermaleng.2014.12.028.

With this information we can calculate the temperatures $T$ at each section
boundary as function of the local $p$ and $h$. For each section the log mean
temperature difference $LMTD_{j}$ can be calculated based on the boundary
temperatures. With $Q_{j}$ the respective $UA_{j}$ of each section is derived
and the overall $UA$ is $\sum UA_{j}$.

With known values for $\alpha_{j}$ for each section of the hot and cold side as
well as conductance resistance it is possible to find the overall heat exchange
area $A$.

$$
A = \frac{\sum \left[UA_{j} \cdot \left(
    
    \frac{1}{\alpha_{\text{h,}j}}
    + \frac{1}{\alpha_{\text{c,}j} \dot A_\text{ratio}}

\right)\right]}{1 - R \cdot \sum UA_{j}}
$$

When operating at conditions different from the design point, the area must be
equal to the design point area. The conductance resistance does not change as
well. $\alpha$ changes, often you can assume it scales with the change of mass
flow:

$$
\alpha = \alpha_\text{design} \cdot \frac{\dot m}{\dot m_\text{design}} ^ {Re_\text{exp}}
$$

Typical Reynolds exponents have been referenced in literature, e.g.

- Cecchinato et al. (2010): "A simpliﬁed method to evaluate the seasonal
  energy performance of water chillers",
  https://doi.org/10.1016/j.ijthermalsci.2010.04.010
- Quoilin (2011): "Sustainable Energy Conversion Through the Use of Organic
  Rankine Cycles for Waste Heat Recovery and Solar Applications",
  https://orbi.uliege.be/bitstream/2268/96436/1/PhD_Thesis_Dissertation.pdf


## LT-ORC (Low-Temperature Organic Rankine Cycle)

This notebook builds and validates the **LT-ORC branch only** (working fluid:
R245fa), coupled to the geothermal water and to the ambient air used to cool
the condenser.


### 1. Imports

In [ ]:
from tespy.networks import Network
from tespy.components import (
    Source, Sink,
    Pump, Turbine,
    SectionedHeatExchanger,
    CycleCloser,
    Splitter, Merge,
    PowerSink, PowerBus,
    Motor, Generator
)
from tespy.connections import Connection, PowerConnection

import numpy as np
import matplotlib.pyplot as plt
from CoolProp.CoolProp import PropsSI

### 2. Network and units

In [ ]:
nw = Network()
nw.units.set_defaults(
    temperature="degC", pressure="bar", pressure_difference="bar",
    enthalpy="kJ/kg", mass_flow="kg/s", power="kW", heat="kW"
)

### 3. Components

In [ ]:
lt_cc = CycleCloser("LT-cycle-closer")

lt_pump = Pump("LT-pump")
lt_preheater = SectionedHeatExchanger("LT-preheater")
lt_evaporator = SectionedHeatExchanger("LT-evaporator")
lt_turbine = Turbine("LT-turbine")
lt_condenser = SectionedHeatExchanger("LT-condenser")

lt_air_source = Source("LT-air-source")
lt_air_sink = Sink("LT-air-sink")

geo_source = Source("geothermal-source")
geo_split = Splitter("geothermal-splitter", num_out=2)
geo_merge = Merge("geothermal-merge", num_in=2)
geo_sink = Sink("geothermal-sink")

lt_generator = Generator("LT-generator")
lt_motor = Motor("LT-motor")

distribution = PowerBus("power-distribution", num_in=1, num_out=2)
grid = PowerSink("grid")

### 4. Connections 

In [ ]:
c7 = Connection(lt_condenser, "out1", lt_pump, "in1", label="07")
c8 = Connection(lt_pump, "out1", lt_preheater, "in2", label="08")
c9 = Connection(lt_preheater, "out2", lt_evaporator, "in2", label="09")
c10 = Connection(lt_evaporator, "out2", lt_turbine, "in1", label="10")
c10a = Connection(lt_turbine, "out1", lt_cc, "in1", label="10a")
c11 = Connection(lt_cc, "out1", lt_condenser, "in1", label="11")

nw.add_conns(c7, c8, c9, c10, c10a, c11)

lt_a1 = Connection(lt_air_source, "out1", lt_condenser, "in2", label="LT-a1")
lt_a2 = Connection(lt_condenser, "out2", lt_air_sink, "in1", label="LT-a2")

nw.add_conns(lt_a1, lt_a2)

# Geothermal

gC = Connection(geo_source, "out1", lt_evaporator, "in1", label="C")
gD1 = Connection(lt_evaporator, "out1", geo_split, "in1", label="D1")
gE = Connection(geo_split, "out1", lt_preheater, "in1", label="E")
gD2= Connection(geo_split, "out2", geo_merge, "in2", label="D2")
gF= Connection(lt_preheater, "out1", geo_merge, "in1", label="F")
gH= Connection(geo_merge, "out1", geo_sink, "in1", label="H")

nw.add_conns(gC, gD1, gD2, gE, gF, gH)

# Power

lt_e1 = PowerConnection(lt_turbine, "power", lt_generator, "power_in", label="LT-e1")
lt_e2 = PowerConnection(lt_generator, "power_out", distribution, "power_in1", label="LT-e2")

lt_e3 = PowerConnection(distribution, "power_out1", lt_motor, "power_in", label="LT-e3")
lt_e4 = PowerConnection(lt_motor, "power_out", lt_pump, "power", label="LT-e4")


e5 = PowerConnection(distribution, "power_out2", grid, "power", label="e5")

nw.add_conns(lt_e1, lt_e2, lt_e3, lt_e4, e5)

### 5. Parameters

In [ ]:
c7.set_attr(fluid={"R245fa": 1}, p=1.55, td_bubble=3)  # condenser outlet subcooled 3 K;
c8.set_attr(p=6.75)
c9.set_attr(td_bubble=2)
c10.set_attr(x=1)

gC.set_attr(fluid={"water": 1}, p=10, m=122.4, T=92.95)
gD1.set_attr(T=70.4)
gD2.set_attr(m=67.9)

lt_a1.set_attr(fluid={"air": 1}, p=1, T=8.67)
lt_a2.set_attr(T=19)

In [ ]:
lt_evaporator.set_attr(dp1=0, dp2=0)
lt_preheater.set_attr(dp2=0.78)  # no dp1 as is parallel to lht path!
lt_condenser.set_attr(dp1=0, dp2=0)

lt_turbine.set_attr(eta_s=0.827)
lt_pump.set_attr(eta_s=0.50)   # paper: isentropic efficiency calculated from design data (50 %)
lt_generator.set_attr(eta=1)
lt_motor.set_attr(eta=1)

### 6. Solve

In [ ]:
nw.solve("design")

## Results

### Overview and key results

In [ ]:
nw.print_results()

In [ ]:
nw.results["Connection"]

In [ ]:
nw.results["PowerConnection"]

In [ ]:
nw.results["SectionedHeatExchanger"]

In [ ]:
W_pump = lt_pump.P.val
W_turbine = lt_turbine.P.val          # negative sign = power output
Q_preheater = lt_preheater.Q.val
Q_evaporator = lt_evaporator.Q.val
Q_condenser = lt_condenser.Q.val

W_net = -W_turbine - W_pump           # kW
Q_in = -(Q_preheater + Q_evaporator)  # kW
eta_th = W_net / Q_in

print("================ LT-ORC SUMMARY ================")
print(f"Pump power         : {W_pump:8.1f} kW")
print(f"Turbine power      : {-W_turbine:8.1f} kW")
print(f"Preheater duty     : {-Q_preheater:8.1f} kW")
print(f"Evaporator duty    : {-Q_evaporator:8.1f} kW")
print(f"Condenser duty     : {-Q_condenser:8.1f} kW")
print(f"Air mass flow rate : {lt_a1.m.val:8.1f} kg/s")
print(f"Net power output   : {W_net:8.1f} kW")
print(f"Thermal efficiency : {eta_th * 100:8.2f} %")

### T-s diagram (LT-ORC)

The plot below shows the LT-ORC on temperature-entropy axes: the R245fa saturation dome, the five
state points (7-10a, matching the flow-chart numbering), and the process paths between them.

In [ ]:
FLUID = "R245fa"

def conn_Ts(conn):
    # T [degC] and s [kJ/kg-K] for a tespy connection (R245fa)
    p_Pa = conn.p.val * 1e5
    h_Jkg = conn.h.val * 1e3
    T = PropsSI("T", "P", p_Pa, "H", h_Jkg, FLUID) - 273.15
    s = PropsSI("S", "P", p_Pa, "H", h_Jkg, FLUID) / 1e3
    return T, s

def isobar_Ts(p_bar, h1_kJkg, h2_kJkg, n=40):
    # T,s along a constant-pressure line between two enthalpies
    p_Pa = p_bar * 1e5
    hs = np.linspace(h1_kJkg, h2_kJkg, n) * 1e3
    T = PropsSI("T", "P", p_Pa, "H", hs, FLUID) - 273.15
    s = PropsSI("S", "P", p_Pa, "H", hs, FLUID) / 1e3
    return T, s

# saturation dome
T_crit = PropsSI("Tcrit", FLUID)
T_dome = np.linspace(280, T_crit - 0.3, 200)
sf = PropsSI("S", "T", T_dome, "Q", 0, FLUID) / 1e3
sg = PropsSI("S", "T", T_dome, "Q", 1, FLUID) / 1e3
T_dome_C = T_dome - 273.15

fig, ax = plt.subplots(figsize=(7, 5.5))
ax.plot(np.concatenate([sf, sg[::-1]]), np.concatenate([T_dome_C, T_dome_C[::-1]]),
        "k-", lw=1, label="sat. dome")

# state points
# 7 = pump inlet, 8 = pump outlet, 9 = preheater outlet / evaporator inlet,
# 10 = evaporator outlet / turbine inlet, 10a = turbine outlet
states = {"7": c7, "8": c8, "9": c9, "10": c10, "10a": c10a}
pts = {k: conn_Ts(v) for k, v in states.items()}

# 7 -> 8: pump (real, non-isentropic process, straight line between end states)
ax.plot([pts["7"][1], pts["8"][1]], [pts["7"][0], pts["8"][0]], "b-", lw=2, label="pump / turbine")
# 8 -> 9: preheater (isobar)
T89, s89 = isobar_Ts(c8.p.val, c8.h.val, c9.h.val)
ax.plot(s89, T89, color="tab:orange", lw=2, label="preheater")
# 9 -> 10: evaporator (isobar, pressure interpolated across preheater -> evaporator outlet)
hs = np.linspace(c9.h.val, c10.h.val, 40) * 1e3
ps = np.linspace(c9.p.val, c10.p.val, 40) * 1e5
T910 = PropsSI("T", "P", ps, "H", hs, FLUID) - 273.15
s910 = PropsSI("S", "P", ps, "H", hs, FLUID) / 1e3
ax.plot(s910, T910, "r-", lw=2, label="evaporator")
# 10 -> 10a: turbine (real, non-isentropic process)
ax.plot([pts["10"][1], pts["10a"][1]], [pts["10"][0], pts["10a"][0]], "b-", lw=2)
# 10a -> 7: condenser (isobar)
T10a7, s10a7 = isobar_Ts(c10a.p.val, c10a.h.val, c7.h.val)
ax.plot(s10a7, T10a7, "c-", lw=2, label="condenser")

for k, (T, s) in pts.items():
    ax.plot(s, T, "ko", ms=5)
    ax.annotate(k, (s, T), textcoords="offset points", xytext=(6, 4))

ax.set_xlabel("specific entropy s [kJ/kg-K]")
ax.set_ylabel("temperature T [degC]")
ax.set_title("LT-ORC T-s diagram (R245fa)")
ax.legend(loc="upper left", fontsize=9)
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

### Pinch point check and heat exchanger T-Q diagrams

These plots show temperature vs. cumulative heat duty for each heat exchanger,
hot and cold streams together, assuming counter-current flow. The vertical gap
between the two curves at any point is the local approach temperature and
should match the `td_pinch` values from the pinch-point check in the cell
below. The deviation is from not including the exact phase change point in the
plots.

In [ ]:
print("Working fluid mass flow rate (SOLVED):", round(c7.m.val, 2), "kg/s")
print()
print(f"{"Component":<15}{"td_pinch [K]":>12}")
for hx in [lt_preheater, lt_evaporator, lt_condenser]:
    flag = "  <-- violation!" if hx.td_pinch.val < 0 else ""
    print(
        f"{hx.label:<15}{hx.td_pinch.val:>12.2f}"
        f"{flag}"
    )

In [ ]:
def counter_current_profile(m_hot, fluid_hot, p_hot_in_bar, p_hot_out_bar, h_hot_in, h_hot_out,
                             m_cold, fluid_cold, p_cold_in_bar, p_cold_out_bar, h_cold_in, h_cold_out,
                             n=40):
    # Q, T_hot, T_cold along a counter-current heat exchanger, evaluated at n
    # equally spaced points of cumulative heat duty Q (starting at the cold-inlet /
    # hot-outlet end, i.e. the coldest end of the exchanger).
    Q_total = m_cold * (h_cold_out - h_cold_in)  # kW
    Q = np.linspace(0, Q_total, n)
    h_cold = h_cold_in + Q / m_cold
    h_hot = h_hot_out + Q / m_hot
    p_cold = np.linspace(p_cold_in_bar, p_cold_out_bar, n) * 1e5
    p_hot = np.linspace(p_hot_out_bar, p_hot_in_bar, n) * 1e5
    T_cold = PropsSI("T", "P", p_cold, "H", h_cold * 1e3, fluid_cold) - 273.15
    T_hot = PropsSI("T", "P", p_hot, "H", h_hot * 1e3, fluid_hot) - 273.15
    return Q, T_hot, T_cold


fig, axs = plt.subplots(1, 3, figsize=(15, 4.5))

# Preheater: hot = geothermal water (E -> F), cold = R245fa (8 -> 9)
Q, Th, Tc = counter_current_profile(
    gE.m.val, "water", gE.p.val, gF.p.val, gE.h.val, gF.h.val,
    c8.m.val, "R245fa", c8.p.val, c9.p.val, c8.h.val, c9.h.val)
axs[0].plot(Q, Th, "r-o", ms=3, label="geothermal water")
axs[0].plot(Q, Tc, "b-o", ms=3, label="R245fa")
axs[0].set_title(f"LT Preheater (min approach = {(Th - Tc).min():.2f} K)")

# Evaporator: hot = geothermal water (C -> D), cold = R245fa (9 -> 10)
Q, Th, Tc = counter_current_profile(
    gC.m.val, "water", gC.p.val, gD1.p.val, gC.h.val, gD1.h.val,
    c9.m.val, "R245fa", c9.p.val, c10.p.val, c9.h.val, c10.h.val)
axs[1].plot(Q, Th, "r-o", ms=3, label="geothermal water")
axs[1].plot(Q, Tc, "b-o", ms=3, label="R245fa")
axs[1].set_title(f"LT Evaporator (min approach = {(Th - Tc).min():.2f} K)")

# Condenser: hot = R245fa (10a -> 7, cooling down), cold = air (7 -> 8)
Q, Th, Tc = counter_current_profile(
    c10a.m.val, "R245fa", c10a.p.val, c7.p.val, c10a.h.val, c7.h.val,
    lt_a1.m.val, "air", lt_a1.p.val, lt_a2.p.val, lt_a1.h.val, lt_a2.h.val)
axs[2].plot(Q, Th, "r-o", ms=3, label="R245fa")
axs[2].plot(Q, Tc, "b-o", ms=3, label="air")
axs[2].set_title(f"LT Condenser (min approach = {(Th - Tc).min():.2f} K)")

for ax in axs:
    ax.set_xlabel("cumulative heat duty Q [kW]")
    ax.set_ylabel("temperature T [degC]")
    ax.grid(alpha=0.3)
    ax.legend()

fig.tight_layout()
plt.show()

### Validation of the design point against Table 3 (Heberle et al., 2015)

The comparison uses the **simulation StanMix column** of Table 3.

In [ ]:
import pandas as pd


# Table 3, simulation StanMix column (* = set variable in this model too).
table3 = {
    "T_C [degC] *":        92.95,
    "T_D [degC] *":        70.37,
    "m_E [kg/s]":          54.48,
    "T_F [degC]":          52.13,
    "T_07 [degC]":         23.03,
    "T_08 [degC]":         23.51,
    "T_09 [degC]":         67.37,
    "T_10 [degC]":         69.37,
    "p_10 [bar]":           5.97,
    "T_10a [degC]":        40.02,
    "m_LT_wf [kg/s]":      69.72,
    "LT feed pump [kW]":   59.76,
}

tespy_design = {
    "T_C [degC] *":      nw.get_conn("C").T.val,
    "T_D [degC] *":      nw.get_conn("D1").T.val,
    "m_E [kg/s]":        nw.get_conn("E").m.val,
    "T_F [degC]":        nw.get_conn("F").T.val,
    "T_07 [degC]":       nw.get_conn("07").T.val,
    "T_08 [degC]":       nw.get_conn("08").T.val,
    "T_09 [degC]":       nw.get_conn("09").T.val,
    "T_10 [degC]":       nw.get_conn("10").T.val,
    "p_10 [bar]":        nw.get_conn("10").p.val,
    "T_10a [degC]":      nw.get_conn("10a").T.val,
    "m_LT_wf [kg/s]":    nw.get_conn("07").m.val,
    "LT feed pump [kW]": nw.get_conn("LT-e4").E.val,
}

validation = pd.DataFrame({"paper (Table 3)": table3, "TESPy": tespy_design})
validation["deviation"] = validation["TESPy"] - validation["paper (Table 3)"]
validation.round(2)